In [0]:
%run ../utils/adls_auth

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
from pyspark.sql.functions import (
    explode, sequence, to_date, col, year, month, dayofmonth,
    dayofweek, date_format, weekofyear, quarter,
)

GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_date"

date_df = (
 
    spark.sql("SELECT sequence(to_date('2021-01-01'), to_date('2026-12-31'), interval 1 day) as date_seq")
    .select(explode(col("date_seq")).alias("full_date"))
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))  # 1=Sunday
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("is_weekend", col("day_of_week").isin(1, 7))
)

date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
print(f"dim_date built: {date_df.count()} rows.")